# 12 — Multimodal Concatenation Baselines

Two tiers of concatenation baselines:

**Tier 1 — Raw concat** (arms 1–4): feature vectors concatenated directly into a shared MLP.

**Tier 2 — Encoder-projected concat** (arms 5–7): each modality projected to 256-d via a
small MLP encoder before concatenation. Controls for encoder projection vs attention in nb13.

| Arm | Modalities | Integration |
|-----|-----------|-------------|
| 1 | rna + protein | raw concat |
| 2 | rna + drug_fp | raw concat |
| 3 | protein + drug_fp | raw concat |
| 4 | rna + protein + drug_fp | raw concat |
| 5 | rna + drug_fp | encoder → concat |
| 6 | protein + drug_fp | encoder → concat |
| 7 | rna + protein + drug_fp | encoder → concat |

## Environment setup (Colab or local)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q torch-geometric rdkit
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## GPU check

In [ ]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cpu':
    print('WARNING: no GPU detected — Runtime > Change runtime type > GPU')

## Imports and config

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import pearsonr, spearmanr, linregress
from sklearn.metrics import r2_score, roc_auc_score
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR       = BASE_PATH / 'data' / 'GDSC2'
PROCESSED_DIR  = BASE_PATH / 'data' / 'processed'
SPLITS_DIR     = BASE_PATH / 'data' / 'splits'
RESULTS_DIR    = BASE_PATH / 'results' / 'concat_multimodal'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'checkpoints').mkdir(exist_ok=True)

COL_CELL_LINE   = 'cell_line_name'
COL_DRUG        = 'drug_name'
COL_IC50        = 'LN_IC50'
COL_CELLOSAURUS = 'cellosaurus_id'
COL_TISSUE      = 'tissue'

TOP_K_FEATURES   = 1000  # applied to RNA and protein independently
VALIDATION_RATIO = 0.1
RANDOM_STATE     = 42

# Set to True to run only fold 0 / LCO for a quick smoke test
QUICK_TEST = True

## Load response pairs and splits

In [ ]:
pairs = pd.read_parquet(PROCESSED_DIR / 'response_pairs.parquet')

with open(SPLITS_DIR / 'splits.json') as f:
    folds = json.load(f)

with open(SPLITS_DIR / 'holdout_groups.json') as f:
    holdout_groups = json.load(f)

print(f'{len(pairs):,} pairs loaded')
for fd in folds:
    print(f"fold {fd['fold']}: train={len(fd['train']):,} | "
          f"lco={len(fd['lco_test']):,} | ldo={len(fd['ldo_test']):,} | "
          f"lto={len(fd['lto_test']):,} | lpo={len(fd['lpo_test']):,}")

## Load omics features and drug fingerprints

In [ ]:
rna         = pd.read_csv(DATA_DIR / 'gene_expression.csv',  index_col=0)
protein     = pd.read_csv(DATA_DIR / 'proteomics.csv',        index_col=0)
drug_smiles = pd.read_csv(DATA_DIR / 'drug_smiles.csv')

# Deduplicate immediately — duplicate cellosaurus_id rows cause silent row-count mismatches
rna     = rna[~rna.index.duplicated(keep='first')].iloc[:, 1:]
protein = protein[~protein.index.duplicated(keep='first')].fillna(0)

OMICS = {'rna': rna, 'protein': protein}

print(f'RNA:     {rna.shape[1]:,} genes    x {rna.shape[0]:,} cell lines')
print(f'Protein: {protein.shape[1]:,} proteins x {protein.shape[0]:,} cell lines')
print(f'Drugs:   {drug_smiles[COL_DRUG].nunique():,}')

In [ ]:
def build_drug_fingerprints(drug_smiles_df: pd.DataFrame,
                             radius: int = 2,
                             n_bits: int = 2048) -> dict:
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = {}
    for _, row in drug_smiles_df.iterrows():
        mol = Chem.MolFromSmiles(row['canonical_smiles'])
        if mol is None:
            continue
        fps[row[COL_DRUG]] = np.array(generator.GetFingerprint(mol), dtype=np.float32)
    return fps

drug_fp = build_drug_fingerprints(drug_smiles)
print(f'Fingerprints: {len(drug_fp):,} drugs | dim={next(iter(drug_fp.values())).shape[0]}')

## Train / validation split

In [ ]:
def make_validation_indices(train_idx: np.ndarray,
                             fraction: float = VALIDATION_RATIO,
                             seed: int = RANDOM_STATE):
    """Hold out a fraction of cell lines, drugs, and tissues from train.
    Same construction as notebook 09 so inner-val sets are comparable."""
    sub = pairs.loc[train_idx]

    def axis_holdout(group_values: pd.Series, seed_offset: int) -> set:
        gss = GroupShuffleSplit(n_splits=1, test_size=fraction,
                                random_state=seed + seed_offset)
        _, val_idx = next(gss.split(np.arange(len(group_values)),
                                    groups=group_values))
        return set(group_values.iloc[val_idx])

    cell_ho   = axis_holdout(sub[COL_CELLOSAURUS], 0)
    drug_ho   = axis_holdout(sub[COL_DRUG],        1)
    tissue_ho = axis_holdout(sub[COL_TISSUE],      2)

    is_val = (
        sub[COL_CELLOSAURUS].isin(cell_ho)
        | sub[COL_DRUG].isin(drug_ho)
        | sub[COL_TISSUE].isin(tissue_ho)
    ).to_numpy()

    return train_idx[~is_val], train_idx[is_val]

train_inner_idx, val_idx = make_validation_indices(np.array(folds[0]['train']))
print(f'train_inner: {len(train_inner_idx):,} | val: {len(val_idx):,}')

## Feature selection

Top-K variance features computed from train cell lines only. Fitted once on the outer train fold and reused for all four arms.

In [ ]:
def select_top_variance_omics(arm: str,
                               train_idx: np.ndarray,
                               k: int) -> pd.Index:
    """Return top-k highest-variance column names for an omics arm.
    Variance is computed over train cell lines only to avoid leakage."""
    train_cells = pairs.loc[train_idx, COL_CELLOSAURUS].unique()
    compact = OMICS[arm].loc[OMICS[arm].index.intersection(train_cells)]
    return compact.var(axis=0).sort_values(ascending=False).index[:k]

top_cols = {
    'rna':     select_top_variance_omics('rna',     np.array(folds[0]['train']), TOP_K_FEATURES),
    'protein': select_top_variance_omics('protein', np.array(folds[0]['train']), TOP_K_FEATURES),
}
print(f"Selected {len(top_cols['rna']):,} RNA features, {len(top_cols['protein']):,} protein features")

## DataLoader helpers

In [ ]:
def build_feature_matrix(idx: np.ndarray, top_cols: dict) -> dict:
    """Build per-modality numpy arrays for a set of pair indices.
    top_cols must be fitted on the train fold before calling on val/test.
    Returns dict with keys 'rna', 'protein', 'drug', 'y'.
    """
    sub   = pairs.loc[idx]
    cells = sub[COL_CELLOSAURUS]

    rna_X     = OMICS['rna'].loc[cells, top_cols['rna']].to_numpy().astype(np.float32)
    protein_X = OMICS['protein'].loc[cells, top_cols['protein']].to_numpy().astype(np.float32)
    drug_X    = np.vstack([drug_fp[d] for d in sub[COL_DRUG]]).astype(np.float32)
    y         = sub[COL_IC50].to_numpy().astype(np.float32)

    assert not np.isnan(rna_X).any(),     'NaNs in RNA'
    assert not np.isnan(protein_X).any(), 'NaNs in protein'
    assert not np.isnan(y).any(),         'NaNs in target'

    return {'rna': rna_X, 'protein': protein_X, 'drug': drug_X, 'y': y}


def make_dataloader(matrices: dict,
                    arm_keys: list,
                    batch_size: int,
                    shuffle: bool) -> DataLoader:
    """arm_keys: subset of ['rna', 'protein', 'drug'] in concatenation order.
    y is always appended as the last tensor.
    """
    tensors = [torch.from_numpy(matrices[k]) for k in arm_keys]
    tensors.append(torch.from_numpy(matrices['y']))
    dataset = TensorDataset(*tensors)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=shuffle)

In [ ]:
# Build feature matrices once — shared across all four arms
print('Building train matrices...')
train_matrices = build_feature_matrix(train_inner_idx, top_cols)
print('Building val matrices...')
val_matrices   = build_feature_matrix(val_idx,          top_cols)

print('Building test matrices...')
lco_matrices = build_feature_matrix(np.array(folds[0]['lco_test']), top_cols)
ldo_matrices = build_feature_matrix(np.array(folds[0]['ldo_test']), top_cols)
lto_matrices = build_feature_matrix(np.array(folds[0]['lto_test']), top_cols)
lpo_matrices = build_feature_matrix(np.array(folds[0]['lpo_test']), top_cols)

print('Done.')
for name, m in [('train', train_matrices), ('val', val_matrices), ('lco', lco_matrices)]:
    print(f"  {name}: rna={m['rna'].shape}, protein={m['protein'].shape}, "
          f"drug={m['drug'].shape}, y={m['y'].shape}")

## `ConcatMLP` architecture

Concatenates any combination of input feature vectors, then passes through a
standard BN → Dropout → ReLU MLP. Identical internal structure to `NN1Omics`
in notebook 09 — `input_dim` is simply `sum(input_dims)`. This is the naive concatenation baseline.

In [ ]:
class ConcatMLP(nn.Module):
    """Naive concatenation multimodal MLP.

    input_dims   : list of ints, one per modality in concatenation order.
    hidden_layers: list of hidden layer widths.
    dropout_prob : applied after BN on every layer except the last hidden layer.
    """

    def __init__(self, input_dims: list, hidden_layers: list, dropout_prob: float = 0.3):
        super().__init__()
        total_dim = sum(input_dims)

        self.fc_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()

        self.fc_layers.append(nn.Linear(total_dim, hidden_layers[0]))
        self.bn_layers.append(nn.BatchNorm1d(hidden_layers[0]))
        for i in range(1, len(hidden_layers)):
            self.fc_layers.append(nn.Linear(hidden_layers[i - 1], hidden_layers[i]))
            self.bn_layers.append(nn.BatchNorm1d(hidden_layers[i]))

        # Final prediction layer — no BN, no dropout (matches NN1Omics in nb 09)
        self.fc_layers.append(nn.Linear(hidden_layers[-1], 1))
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, *inputs):
        x = torch.cat(inputs, dim=1)
        for i in range(len(self.fc_layers) - 2):
            x = self.fc_layers[i](x)
            x = self.bn_layers[i](x)
            x = self.dropout(x)
            x = F.relu(x)
        x = F.relu(self.fc_layers[-2](x))  # last hidden: no BN/dropout
        return self.fc_layers[-1](x).squeeze(-1)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Training loop

In [ ]:
def fit_concat_mlp(model: nn.Module,
                   train_loader: DataLoader,
                   val_loader: DataLoader,
                   arm_keys: list,
                   num_epochs: int,
                   patience: int,
                   checkpoint_path: Path,
                   device: torch.device):
    """arm_keys: modality keys in DataLoader tensor order (y is always the last tensor)."""
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    history = {'train_loss': [], 'train_cor': [], 'val_loss': [], 'val_cor': []}
    best_val_loss = float('inf')
    best_weights  = None
    wait          = 0

    print(f"Training arm: {' + '.join(arm_keys)} | params: {count_parameters(model):,}")

    for epoch in range(num_epochs):
        for phase in ['val', 'train']:
            loader = train_loader if phase == 'train' else val_loader
            phase_device = device if phase == 'train' else torch.device('cpu')
            model.to(phase_device)
            model.train() if phase == 'train' else model.eval()

            batch_losses, preds, targets = [], [], []

            for batch in tqdm(loader, desc=f'epoch {epoch+1} {phase}', leave=False):
                # Last tensor in batch is y; all preceding tensors are modality features
                *modality_tensors, y = [t.to(phase_device) for t in batch]

                if phase == 'train':
                    optimizer.zero_grad()
                    pred = model(*modality_tensors)
                    loss = criterion(pred, y)
                    loss.backward()
                    optimizer.step()
                else:
                    with torch.no_grad():
                        pred = model(*modality_tensors)
                        loss = criterion(pred, y)

                batch_losses.append(loss.item())
                preds.append(pred.detach())
                targets.append(y.detach())

            epoch_loss = sum(batch_losses) / len(batch_losses)
            all_preds   = torch.cat(preds).cpu().numpy()
            all_targets = torch.cat(targets).cpu().numpy()
            epoch_cor, _ = pearsonr(all_targets, all_preds)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_cor'].append(epoch_cor)

            if phase == 'val':
                print(f'  epoch {epoch+1:>3} | val loss={epoch_loss:.4f}  pearson_r={epoch_cor:.4f}')
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_weights  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f'  Early stopping at epoch {epoch + 1}')
                        model.load_state_dict(best_weights)
                        torch.save(best_weights, checkpoint_path)
                        print(f'  Best val loss: {best_val_loss:.4f} -> saved to {checkpoint_path}')
                        return model, history

    model.load_state_dict(best_weights)
    torch.save(best_weights, checkpoint_path)
    print(f'  Best val loss: {best_val_loss:.4f} -> saved to {checkpoint_path}')
    return model, history

## Prediction and evaluation helpers

In [ ]:
def predict_concat(model: nn.Module,
                   matrices: dict,
                   arm_keys: list,
                   batch_size: int = 512,
                   device: str = 'cpu') -> np.ndarray:
    """Run inference on a pre-built feature matrix dict.
    Returns predictions as a 1-D numpy array.
    """
    model.to(device)
    model.eval()
    tensors = [torch.from_numpy(matrices[k]) for k in arm_keys]
    n = tensors[0].shape[0]
    preds = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch_inputs = [t[i:i + batch_size].to(device) for t in tensors]
            preds.append(model(*batch_inputs).cpu())
    return torch.cat(preds).numpy()

In [ ]:
def evaluate_split(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Returns a flat dict of metrics for one split."""
    corr,  corr_p  = pearsonr(y_true, y_pred)
    spear, spear_p = spearmanr(y_true, y_pred)
    mse            = float(np.mean((y_true - y_pred) ** 2))
    rmse           = float(np.sqrt(mse))
    r2             = float(r2_score(y_true, y_pred))
    slope, _, _, _, std_err = linregress(y_pred, y_true)
    threshold = np.median(y_true)
    binary    = (y_true < threshold).astype(int)
    roc = roc_auc_score(binary, -y_pred) if len(np.unique(binary)) == 2 else np.nan
    return {
        'Pearson r':  round(corr,     4),
        'Spearman r': round(spear,    4),
        'RMSE':       round(rmse,     4),
        'R2':         round(r2,       4),
        'ROC-AUC':    round(roc, 4) if not np.isnan(roc) else roc,
        'Slope':      round(slope,    4),
        'Std err':    round(std_err,  4),
    }


def evaluate_all_splits(arm_name: str, preds: dict) -> pd.DataFrame:
    """preds: dict of split_name -> (y_true, y_pred).
    For 'LPO', the full array is evaluated as 'LPO - all', then each
    lpo_masks sub-split is evaluated separately.
    """
    rows = []
    for split_name, (y_true, y_pred) in preds.items():
        if split_name == 'LPO':
            # Overall LPO
            row = {'arm': arm_name, 'split': 'LPO - all'}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)
            # Sub-splits by novelty
            for mask_name, mask in lpo_masks.items():
                if mask.sum() == 0:
                    continue
                row = {'arm': arm_name, 'split': mask_name}
                row.update(evaluate_split(y_true[mask], y_pred[mask]))
                rows.append(row)
        else:
            row = {'arm': arm_name, 'split': split_name}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def build_lpo_masks(fold_idx: int = 0) -> dict:
    cell_holdout = set(holdout_groups[fold_idx]['cell_lines_held_out'])
    drug_holdout = set(holdout_groups[fold_idx]['drugs_held_out'])
    lpo_idx = np.array(folds[fold_idx]['lpo_test'])
    sub = pairs.loc[lpo_idx]
    is_new_cell = sub[COL_CELLOSAURUS].isin(cell_holdout).to_numpy()
    is_new_drug = sub[COL_DRUG].isin(drug_holdout).to_numpy()
    masks = {
        'LPO - nothing new':           (~is_new_cell) & (~is_new_drug),
        'LPO - new cell line only':     is_new_cell   & (~is_new_drug),
        'LPO - new drug only':          (~is_new_cell) & is_new_drug,
        'LPO - fully new (cell+drug)':  is_new_cell   & is_new_drug,
    }
    for name, mask in masks.items():
        print(f'{name}: n={mask.sum():,}')
    return masks

lpo_masks = build_lpo_masks(fold_idx=0)

## Shared hyperparameters

In [37]:
HIDDEN_LAYERS = [512, 256]
DROPOUT       = 0.3
BATCH_SIZE    = 64
NUM_EPOCHS    = 20
PATIENCE      = 10

DIM_RNA     = train_matrices['rna'].shape[1]
DIM_PROTEIN = train_matrices['protein'].shape[1]
DIM_DRUG_FP = train_matrices['drug'].shape[1]

print(f'RNA dim: {DIM_RNA} | Protein dim: {DIM_PROTEIN} | Drug FP dim: {DIM_DRUG_FP}')

RNA dim: 1000 | Protein dim: 1000 | Drug FP dim: 2048


---
## Arm 1 — RNA + Protein

In [ ]:
ARM1_KEYS = ['rna', 'protein']
arm1_train_loader = make_dataloader(train_matrices, ARM1_KEYS, BATCH_SIZE, shuffle=True)
arm1_val_loader   = make_dataloader(val_matrices,   ARM1_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm1_model = ConcatMLP(
    input_dims=[DIM_RNA, DIM_PROTEIN],
    hidden_layers=HIDDEN_LAYERS,
    dropout_prob=DROPOUT,
)
print(f'Arm 1 (rna+protein) parameters: {count_parameters(arm1_model):,}')

In [ ]:
arm1_model, arm1_history = fit_concat_mlp(
    model=arm1_model,
    train_loader=arm1_train_loader,
    val_loader=arm1_val_loader,
    arm_keys=ARM1_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm1_rna_protein.pt',
    device=DEVICE,
)

In [ ]:
arm1_preds = {
    'LCO': (lco_matrices['y'], predict_concat(arm1_model, lco_matrices, ARM1_KEYS)),
    'LDO': (ldo_matrices['y'], predict_concat(arm1_model, ldo_matrices, ARM1_KEYS)),
    'LTO': (lto_matrices['y'], predict_concat(arm1_model, lto_matrices, ARM1_KEYS)),
    'LPO': (lpo_matrices['y'], predict_concat(arm1_model, lpo_matrices, ARM1_KEYS)),
}
arm1_results = evaluate_all_splits('rna+protein', arm1_preds)
arm1_results

---
## Arm 2 — RNA + Drug FP

In [ ]:
ARM2_KEYS = ['rna', 'drug']
arm2_train_loader = make_dataloader(train_matrices, ARM2_KEYS, BATCH_SIZE, shuffle=True)
arm2_val_loader   = make_dataloader(val_matrices,   ARM2_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm2_model = ConcatMLP(
    input_dims=[DIM_RNA, DIM_DRUG_FP],
    hidden_layers=HIDDEN_LAYERS,
    dropout_prob=DROPOUT,
)
print(f'Arm 2 (rna+drug_fp) parameters: {count_parameters(arm2_model):,}')

In [ ]:
arm2_model, arm2_history = fit_concat_mlp(
    model=arm2_model,
    train_loader=arm2_train_loader,
    val_loader=arm2_val_loader,
    arm_keys=ARM2_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm2_rna_drug.pt',
    device=DEVICE,
)

In [ ]:
arm2_preds = {
    'LCO': (lco_matrices['y'], predict_concat(arm2_model, lco_matrices, ARM2_KEYS)),
    'LDO': (ldo_matrices['y'], predict_concat(arm2_model, ldo_matrices, ARM2_KEYS)),
    'LTO': (lto_matrices['y'], predict_concat(arm2_model, lto_matrices, ARM2_KEYS)),
    'LPO': (lpo_matrices['y'], predict_concat(arm2_model, lpo_matrices, ARM2_KEYS)),
}
arm2_results = evaluate_all_splits('rna+drug_fp', arm2_preds)
arm2_results

---
## Arm 3 — Protein + Drug FP

In [ ]:
ARM3_KEYS = ['protein', 'drug']
arm3_train_loader = make_dataloader(train_matrices, ARM3_KEYS, BATCH_SIZE, shuffle=True)
arm3_val_loader   = make_dataloader(val_matrices,   ARM3_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm3_model = ConcatMLP(
    input_dims=[DIM_PROTEIN, DIM_DRUG_FP],
    hidden_layers=HIDDEN_LAYERS,
    dropout_prob=DROPOUT,
)
print(f'Arm 3 (protein+drug_fp) parameters: {count_parameters(arm3_model):,}')

In [ ]:
arm3_model, arm3_history = fit_concat_mlp(
    model=arm3_model,
    train_loader=arm3_train_loader,
    val_loader=arm3_val_loader,
    arm_keys=ARM3_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm3_protein_drug.pt',
    device=DEVICE,
)

In [ ]:
arm3_preds = {
    'LCO': (lco_matrices['y'], predict_concat(arm3_model, lco_matrices, ARM3_KEYS)),
    'LDO': (ldo_matrices['y'], predict_concat(arm3_model, ldo_matrices, ARM3_KEYS)),
    'LTO': (lto_matrices['y'], predict_concat(arm3_model, lto_matrices, ARM3_KEYS)),
    'LPO': (lpo_matrices['y'], predict_concat(arm3_model, lpo_matrices, ARM3_KEYS)),
}
arm3_results = evaluate_all_splits('protein+drug_fp', arm3_preds)
arm3_results

---
## Arm 4 — RNA + Protein + Drug FP

In [ ]:
ARM4_KEYS = ['rna', 'protein', 'drug']
arm4_train_loader = make_dataloader(train_matrices, ARM4_KEYS, BATCH_SIZE, shuffle=True)
arm4_val_loader   = make_dataloader(val_matrices,   ARM4_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm4_model = ConcatMLP(
    input_dims=[DIM_RNA, DIM_PROTEIN, DIM_DRUG_FP],
    hidden_layers=HIDDEN_LAYERS,
    dropout_prob=DROPOUT,
)
print(f'Arm 4 (rna+protein+drug_fp) parameters: {count_parameters(arm4_model):,}')

In [ ]:
arm4_model, arm4_history = fit_concat_mlp(
    model=arm4_model,
    train_loader=arm4_train_loader,
    val_loader=arm4_val_loader,
    arm_keys=ARM4_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm4_rna_protein_drug.pt',
    device=DEVICE,
)

In [ ]:
arm4_preds = {
    'LCO': (lco_matrices['y'], predict_concat(arm4_model, lco_matrices, ARM4_KEYS)),
    'LDO': (ldo_matrices['y'], predict_concat(arm4_model, ldo_matrices, ARM4_KEYS)),
    'LTO': (lto_matrices['y'], predict_concat(arm4_model, lto_matrices, ARM4_KEYS)),
    'LPO': (lpo_matrices['y'], predict_concat(arm4_model, lpo_matrices, ARM4_KEYS)),
}
arm4_results = evaluate_all_splits('rna+protein+drug_fp', arm4_preds)
arm4_results

---
## Encoder-projected arms (ablation)

Arms 5–7 project each modality through a small MLP encoder to 256-d **before**
concatenation, then feed the concatenated embeddings into a lightweight predictor MLP.
This isolates the effect of encoder projection from the attention mechanism in nb13:
if these arms improve over arms 2–4, the gain is from projection alone, not attention.

Architecture per modality:
`input → Linear(input_dim, 512) → BN → Dropout → ReLU → Linear(512, 256) → BN → ReLU → 256-d embedding`

Predictor after concat:
`concat_dim → Linear → BN → Dropout → ReLU → Linear → 1`

In [38]:
class OmicsEncoder(nn.Module):
    """Projects a single omics modality to a fixed 256-d embedding."""

    def __init__(self, input_dim: int, hidden: int = 512, out_dim: int = 256,
                 dropout_prob: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.Dropout(dropout_prob),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DrugMLPEncoder(nn.Module):
    """Projects drug fingerprint (2048-d) to 256-d embedding.
    Same structure as OmicsEncoder — keeps all modalities in the same space
    before concatenation so no single modality dominates by raw feature count.
    """

    def __init__(self, input_dim: int = 2048, hidden: int = 512, out_dim: int = 256,
                 dropout_prob: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.Dropout(dropout_prob),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EncodedConcatMLP(nn.Module):
    """Encode each modality to 256-d, concatenate, then predict.

    encoders : list of nn.Module, one per modality in input order.
    n_modalities : number of modalities (sets predictor input dim = n * 256).
    """

    def __init__(self, encoders: list, n_modalities: int,
                 embed_dim: int = 256, dropout_prob: float = 0.3):
        super().__init__()
        self.encoders = nn.ModuleList(encoders)
        concat_dim = embed_dim * n_modalities
        self.predictor = nn.Sequential(
            nn.Linear(concat_dim, concat_dim // 2),
            nn.BatchNorm1d(concat_dim // 2),
            nn.Dropout(dropout_prob),
            nn.ReLU(),
            nn.Linear(concat_dim // 2, 1),
        )

    def forward(self, *inputs):
        embeddings = [enc(x) for enc, x in zip(self.encoders, inputs)]
        return self.predictor(torch.cat(embeddings, dim=1)).squeeze(-1)

In [39]:
class OmicsFusionMLP(nn.Module):
    """Encode RNA and protein to 256-d, fuse omics with hadamard or gated fusion,
    then concat with drug embedding and predict.

    fusion_mode:
        'hadamard' — element-wise product of rna_emb and protein_emb.
                     Amplifies dimensions where both modalities are active;
                     suppresses dimensions where one is near zero.
                     No extra parameters.
        'gated'    — learned per-dimension soft switch.
                     gate = sigmoid(W_r·rna_emb + W_p·protein_emb)
                     fused = gate ⊙ rna_emb + (1 - gate) ⊙ protein_emb
                     Learns which latent dimensions trust RNA vs protein.
    """

    def __init__(self, rna_dim: int, protein_dim: int, drug_dim: int,
                 embed_dim: int = 256, fusion_mode: str = 'hadamard',
                 dropout_prob: float = 0.3):
        super().__init__()
        assert fusion_mode in ('hadamard', 'gated')
        self.fusion_mode = fusion_mode

        self.rna_encoder     = OmicsEncoder(rna_dim,     dropout_prob=dropout_prob)
        self.protein_encoder = OmicsEncoder(protein_dim, dropout_prob=dropout_prob)
        self.drug_encoder    = DrugMLPEncoder(drug_dim,  dropout_prob=dropout_prob)

        if fusion_mode == 'gated':
            # Two linear layers produce a per-dimension gate in [0, 1]
            self.gate_rna     = nn.Linear(embed_dim, embed_dim, bias=False)
            self.gate_protein = nn.Linear(embed_dim, embed_dim, bias=False)

        # Predictor: concat(omics_fused, drug_emb) → scalar
        concat_dim = embed_dim + embed_dim  # fused omics (256) + drug (256)
        self.predictor = nn.Sequential(
            nn.Linear(concat_dim, concat_dim // 2),
            nn.BatchNorm1d(concat_dim // 2),
            nn.Dropout(dropout_prob),
            nn.ReLU(),
            nn.Linear(concat_dim // 2, 1),
        )

    def forward(self, rna: torch.Tensor, protein: torch.Tensor,
                drug: torch.Tensor) -> torch.Tensor:
        rna_emb     = self.rna_encoder(rna)
        protein_emb = self.protein_encoder(protein)
        drug_emb    = self.drug_encoder(drug)

        if self.fusion_mode == 'hadamard':
            # Element-wise product — 256-d, no extra params
            omics_fused = rna_emb * protein_emb
        else:
            # Gated fusion — learned per-dimension trust of RNA vs protein
            gate = torch.sigmoid(self.gate_rna(rna_emb) + self.gate_protein(protein_emb))
            omics_fused = gate * rna_emb + (1.0 - gate) * protein_emb

        return self.predictor(torch.cat([omics_fused, drug_emb], dim=1)).squeeze(-1)


def fit_fusion_mlp(model: nn.Module,
                   train_loader: DataLoader,
                   val_loader: DataLoader,
                   arm_name: str,
                   num_epochs: int,
                   patience: int,
                   checkpoint_path: Path,
                   device: torch.device):
    """Training loop for OmicsFusionMLP.
    Batch order: (rna, protein, drug, y) — uses ARM_ALL_KEYS.
    Same loop as fit_encoded_mlp; kept separate for independent execution.
    """
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    history = {'train_loss': [], 'train_cor': [], 'val_loss': [], 'val_cor': []}
    best_val_loss = float('inf')
    best_weights  = None
    wait          = 0

    print(f'Training arm: {arm_name} | params: {count_parameters(model):,}')

    for epoch in range(num_epochs):
        for phase in ['val', 'train']:
            loader       = train_loader if phase == 'train' else val_loader
            phase_device = device if phase == 'train' else torch.device('cpu')
            model.to(phase_device)
            model.train() if phase == 'train' else model.eval()

            batch_losses, preds, targets = [], [], []

            for batch in tqdm(loader, desc=f'epoch {epoch+1} {phase}', leave=False):
                rna, protein, drug, y = [t.to(phase_device) for t in batch]

                if phase == 'train':
                    optimizer.zero_grad()
                    pred = model(rna, protein, drug)
                    loss = criterion(pred, y)
                    loss.backward()
                    optimizer.step()
                else:
                    with torch.no_grad():
                        pred = model(rna, protein, drug)
                        loss = criterion(pred, y)

                batch_losses.append(loss.item())
                preds.append(pred.detach())
                targets.append(y.detach())

            epoch_loss = sum(batch_losses) / len(batch_losses)
            epoch_cor, _ = pearsonr(
                torch.cat(targets).cpu().numpy(),
                torch.cat(preds).cpu().numpy(),
            )
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_cor'].append(epoch_cor)

            if phase == 'val':
                print(f'  epoch {epoch+1:>3} | val loss={epoch_loss:.4f}  pearson_r={epoch_cor:.4f}')
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_weights  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f'  Early stopping at epoch {epoch+1}')
                        model.load_state_dict(best_weights)
                        torch.save(best_weights, checkpoint_path)
                        print(f'  Best val loss: {best_val_loss:.4f} -> {checkpoint_path}')
                        return model, history

    model.load_state_dict(best_weights)
    torch.save(best_weights, checkpoint_path)
    print(f'  Best val loss: {best_val_loss:.4f} -> {checkpoint_path}')
    return model, history


def predict_fusion(model: nn.Module, matrices: dict,
                   batch_size: int = 512, device: str = 'cpu') -> np.ndarray:
    """Inference for OmicsFusionMLP. matrices must have keys rna, protein, drug."""
    model.to(device)
    model.eval()
    rna_t     = torch.from_numpy(matrices['rna'])
    protein_t = torch.from_numpy(matrices['protein'])
    drug_t    = torch.from_numpy(matrices['drug'])
    n = rna_t.shape[0]
    preds = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            preds.append(model(
                rna_t[i:i+batch_size].to(device),
                protein_t[i:i+batch_size].to(device),
                drug_t[i:i+batch_size].to(device),
            ).cpu())
    return torch.cat(preds).numpy()

In [40]:
def fit_encoded_mlp(model: nn.Module,
                    train_loader: DataLoader,
                    val_loader: DataLoader,
                    arm_keys: list,
                    num_epochs: int,
                    patience: int,
                    checkpoint_path: Path,
                    device: torch.device):
    """Same loop as fit_concat_mlp — reused here without change.
    Kept separate so encoder arms are independently runnable.
    """
    criterion = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    history = {'train_loss': [], 'train_cor': [], 'val_loss': [], 'val_cor': []}
    best_val_loss = float('inf')
    best_weights  = None
    wait          = 0

    print(f"Training arm: {' + '.join(arm_keys)} | params: {count_parameters(model):,}")

    for epoch in range(num_epochs):
        for phase in ['val', 'train']:
            loader = train_loader if phase == 'train' else val_loader
            phase_device = device if phase == 'train' else torch.device('cpu')
            model.to(phase_device)
            model.train() if phase == 'train' else model.eval()

            batch_losses, preds, targets = [], [], []

            for batch in tqdm(loader, desc=f'epoch {epoch+1} {phase}', leave=False):
                *modality_tensors, y = [t.to(phase_device) for t in batch]

                if phase == 'train':
                    optimizer.zero_grad()
                    pred = model(*modality_tensors)
                    loss = criterion(pred, y)
                    loss.backward()
                    optimizer.step()
                else:
                    with torch.no_grad():
                        pred = model(*modality_tensors)
                        loss = criterion(pred, y)

                batch_losses.append(loss.item())
                preds.append(pred.detach())
                targets.append(y.detach())

            epoch_loss = sum(batch_losses) / len(batch_losses)
            all_preds   = torch.cat(preds).cpu().numpy()
            all_targets = torch.cat(targets).cpu().numpy()
            epoch_cor, _ = pearsonr(all_targets, all_preds)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_cor'].append(epoch_cor)

            if phase == 'val':
                print(f'  epoch {epoch+1:>3} | val loss={epoch_loss:.4f}  pearson_r={epoch_cor:.4f}')
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_weights  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f'  Early stopping at epoch {epoch + 1}')
                        model.load_state_dict(best_weights)
                        torch.save(best_weights, checkpoint_path)
                        print(f'  Best val loss: {best_val_loss:.4f} -> saved to {checkpoint_path}')
                        return model, history

    model.load_state_dict(best_weights)
    torch.save(best_weights, checkpoint_path)
    print(f'  Best val loss: {best_val_loss:.4f} -> saved to {checkpoint_path}')
    return model, history


def predict_encoded(model: nn.Module,
                    matrices: dict,
                    arm_keys: list,
                    batch_size: int = 512,
                    device: str = 'cpu') -> np.ndarray:
    """Inference for EncodedConcatMLP — same interface as predict_concat."""
    model.to(device)
    model.eval()
    tensors = [torch.from_numpy(matrices[k]) for k in arm_keys]
    n = tensors[0].shape[0]
    preds = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch_inputs = [t[i:i + batch_size].to(device) for t in tensors]
            preds.append(model(*batch_inputs).cpu())
    return torch.cat(preds).numpy()

---
## Arm 5 — RNA encoder + Drug FP encoder

In [ ]:
ARM5_KEYS = ['rna', 'drug']
arm5_train_loader = make_dataloader(train_matrices, ARM5_KEYS, BATCH_SIZE, shuffle=True)
arm5_val_loader   = make_dataloader(val_matrices,   ARM5_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm5_model = EncodedConcatMLP(
    encoders=[OmicsEncoder(DIM_RNA), DrugMLPEncoder(DIM_DRUG_FP)],
    n_modalities=2,
)
print(f'Arm 5 (rna_enc+drug_enc) parameters: {count_parameters(arm5_model):,}')

In [ ]:
arm5_model, arm5_history = fit_encoded_mlp(
    model=arm5_model,
    train_loader=arm5_train_loader,
    val_loader=arm5_val_loader,
    arm_keys=ARM5_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm5_rna_enc_drug_enc.pt',
    device=DEVICE,
)

In [ ]:
arm5_preds = {
    'LCO': (lco_matrices['y'], predict_encoded(arm5_model, lco_matrices, ARM5_KEYS)),
    'LDO': (ldo_matrices['y'], predict_encoded(arm5_model, ldo_matrices, ARM5_KEYS)),
    'LTO': (lto_matrices['y'], predict_encoded(arm5_model, lto_matrices, ARM5_KEYS)),
    'LPO': (lpo_matrices['y'], predict_encoded(arm5_model, lpo_matrices, ARM5_KEYS)),
}
arm5_results = evaluate_all_splits('rna_enc+drug_enc', arm5_preds)
arm5_results

---
## Arm 6 — Protein encoder + Drug FP encoder

In [ ]:
ARM6_KEYS = ['protein', 'drug']
arm6_train_loader = make_dataloader(train_matrices, ARM6_KEYS, BATCH_SIZE, shuffle=True)
arm6_val_loader   = make_dataloader(val_matrices,   ARM6_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm6_model = EncodedConcatMLP(
    encoders=[OmicsEncoder(DIM_PROTEIN), DrugMLPEncoder(DIM_DRUG_FP)],
    n_modalities=2,
)
print(f'Arm 6 (protein_enc+drug_enc) parameters: {count_parameters(arm6_model):,}')

In [ ]:
arm6_model, arm6_history = fit_encoded_mlp(
    model=arm6_model,
    train_loader=arm6_train_loader,
    val_loader=arm6_val_loader,
    arm_keys=ARM6_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm6_protein_enc_drug_enc.pt',
    device=DEVICE,
)

In [ ]:
arm6_preds = {
    'LCO': (lco_matrices['y'], predict_encoded(arm6_model, lco_matrices, ARM6_KEYS)),
    'LDO': (ldo_matrices['y'], predict_encoded(arm6_model, ldo_matrices, ARM6_KEYS)),
    'LTO': (lto_matrices['y'], predict_encoded(arm6_model, lto_matrices, ARM6_KEYS)),
    'LPO': (lpo_matrices['y'], predict_encoded(arm6_model, lpo_matrices, ARM6_KEYS)),
}
arm6_results = evaluate_all_splits('protein_enc+drug_enc', arm6_preds)
arm6_results

---
## Arm 7 — RNA encoder + Protein encoder + Drug FP encoder

In [ ]:
ARM7_KEYS = ['rna', 'protein', 'drug']
arm7_train_loader = make_dataloader(train_matrices, ARM7_KEYS, BATCH_SIZE, shuffle=True)
arm7_val_loader   = make_dataloader(val_matrices,   ARM7_KEYS, BATCH_SIZE, shuffle=False)

In [ ]:
arm7_model = EncodedConcatMLP(
    encoders=[OmicsEncoder(DIM_RNA), OmicsEncoder(DIM_PROTEIN), DrugMLPEncoder(DIM_DRUG_FP)],
    n_modalities=3,
)
print(f'Arm 7 (rna_enc+protein_enc+drug_enc) parameters: {count_parameters(arm7_model):,}')

In [ ]:
arm7_model, arm7_history = fit_encoded_mlp(
    model=arm7_model,
    train_loader=arm7_train_loader,
    val_loader=arm7_val_loader,
    arm_keys=ARM7_KEYS,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm7_rna_enc_protein_enc_drug_enc.pt',
    device=DEVICE,
)

In [ ]:
arm7_preds = {
    'LCO': (lco_matrices['y'], predict_encoded(arm7_model, lco_matrices, ARM7_KEYS)),
    'LDO': (ldo_matrices['y'], predict_encoded(arm7_model, ldo_matrices, ARM7_KEYS)),
    'LTO': (lto_matrices['y'], predict_encoded(arm7_model, lto_matrices, ARM7_KEYS)),
    'LPO': (lpo_matrices['y'], predict_encoded(arm7_model, lpo_matrices, ARM7_KEYS)),
}
arm7_results = evaluate_all_splits('rna_enc+protein_enc+drug_enc', arm7_preds)
arm7_results

---
## Arm 8 — Hadamard omics fusion + Drug FP encoder

`rna_emb ⊙ protein_emb → 256-d | concat drug_emb → 512-d → predictor`

Element-wise product amplifies dimensions where both modalities are active and suppresses dimensions where one is near zero. No extra parameters over arm 7.

In [41]:
# Shared dataloader for arms 8 & 9 — same key order as arm 7
ARM_ALL_KEYS = ['rna', 'protein', 'drug']
arm89_train_loader = make_dataloader(train_matrices, ARM_ALL_KEYS, BATCH_SIZE, shuffle=True)
arm89_val_loader   = make_dataloader(val_matrices,   ARM_ALL_KEYS, BATCH_SIZE, shuffle=False)

In [42]:
arm8_model = OmicsFusionMLP(
    rna_dim=DIM_RNA, protein_dim=DIM_PROTEIN, drug_dim=DIM_DRUG_FP,
    fusion_mode='hadamard',
)
print(f'Arm 8 (hadamard) parameters: {count_parameters(arm8_model):,}')
print('  Note: same param count as arm 7 — hadamard adds no parameters')

Arm 8 (hadamard) parameters: 2,604,801
  Note: same param count as arm 7 — hadamard adds no parameters


In [43]:
arm8_model, arm8_history = fit_fusion_mlp(
    model=arm8_model,
    train_loader=arm89_train_loader,
    val_loader=arm89_val_loader,
    arm_name='hadamard omics + drug_enc',
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm8_hadamard.pt',
    device=DEVICE,
)

Training arm: hadamard omics + drug_enc | params: 2,604,801


epoch 1 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch   1 | val loss=3.2151  pearson_r=-0.1401


epoch 1 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 2 val:   0%|          | 0/442 [00:00<?, ?it/s]

...

epoch 19 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 20 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch  20 | val loss=0.8866  pearson_r=0.7244


epoch 20 train:   0%|          | 0/1237 [00:00<?, ?it/s]

  Best val loss: 0.8594 -> /content/drive/MyDrive/multiomics_project/results/concat_multimodal/checkpoints/arm8_hadamard.pt


In [44]:
arm8_preds = {
    'LCO': (lco_matrices['y'], predict_fusion(arm8_model, lco_matrices)),
    'LDO': (ldo_matrices['y'], predict_fusion(arm8_model, ldo_matrices)),
    'LTO': (lto_matrices['y'], predict_fusion(arm8_model, lto_matrices)),
    'LPO': (lpo_matrices['y'], predict_fusion(arm8_model, lpo_matrices)),
}
arm8_results = evaluate_all_splits('hadamard+drug_enc', arm8_preds)
arm8_results

,arm,split,Pearson r,Spearman r,RMSE,R2,ROC-AUC,Slope,Std err
0,hadamard+drug_enc,LCO,0.8005,0.7536,1.6169,0.6351,0.8761,0.9425,0.0053
1,hadamard+drug_enc,LDO,0.4408,0.4293,2.3564,0.1484,0.7007,1.0987,0.0165
2,hadamard+drug_enc,LTO,0.8111,0.7682,1.6236,0.6506,0.8796,0.9525,0.0043
3,hadamard+drug_enc,LPO - all,0.8405,0.8070,1.4686,0.7033,0.9002,0.9705,0.0044
4,hadamard+drug_enc,LPO - nothing new,0.8762,0.8482,1.3158,0.7666,0.9217,0.9754,0.0042
5,hadamard+drug_enc,LPO - new cell line only,0.8350,0.7793,1.4835,0.6924,0.8885,0.9425,0.0146
6,hadamard+drug_enc,LPO - new drug only,0.4570,0.4456,2.2868,0.1477,0.7122,1.1301,0.0496
7,hadamard+drug_enc,LPO - fully new (cell+drug),0.3934,0.3768,2.5233,0.0948,0.6771,1.2462,0.1972


---
## Arm 9 — Gated omics fusion + Drug FP encoder

`gate = σ(W_r·rna_emb + W_p·protein_emb)`
`fused = gate ⊙ rna_emb + (1 - gate) ⊙ protein_emb → 256-d | concat drug_emb → 512-d → predictor`

Learned per-dimension soft switch: each of the 256 latent dimensions independently learns how much to trust RNA vs protein. Adds 2 × 256×256 = 131,072 parameters over arm 7.

In [45]:
arm9_model = OmicsFusionMLP(
    rna_dim=DIM_RNA, protein_dim=DIM_PROTEIN, drug_dim=DIM_DRUG_FP,
    fusion_mode='gated',
)
print(f'Arm 9 (gated) parameters: {count_parameters(arm9_model):,}')

Arm 9 (gated) parameters: 2,735,873


In [46]:
arm9_model, arm9_history = fit_fusion_mlp(
    model=arm9_model,
    train_loader=arm89_train_loader,
    val_loader=arm89_val_loader,
    arm_name='gated omics + drug_enc',
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    checkpoint_path=RESULTS_DIR / 'checkpoints' / 'arm9_gated.pt',
    device=DEVICE,
)

Training arm: gated omics + drug_enc | params: 2,735,873


epoch 1 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch   1 | val loss=3.1990  pearson_r=-0.0317


epoch 1 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 2 val:   0%|          | 0/442 [00:00<?, ?it/s]

...

epoch 19 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch  19 | val loss=0.8751  pearson_r=0.7301


epoch 19 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 20 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch  20 | val loss=0.8642  pearson_r=0.7343
  Early stopping at epoch 20
  Best val loss: 0.8586 -> /content/drive/MyDrive/multiomics_project/results/concat_multimodal/checkpoints/arm9_gated.pt


In [47]:
arm9_preds = {
    'LCO': (lco_matrices['y'], predict_fusion(arm9_model, lco_matrices)),
    'LDO': (ldo_matrices['y'], predict_fusion(arm9_model, ldo_matrices)),
    'LTO': (lto_matrices['y'], predict_fusion(arm9_model, lto_matrices)),
    'LPO': (lpo_matrices['y'], predict_fusion(arm9_model, lpo_matrices)),
}
arm9_results = evaluate_all_splits('gated+drug_enc', arm9_preds)
arm9_results

,arm,split,Pearson r,Spearman r,RMSE,R2,ROC-AUC,Slope,Std err
0,gated+drug_enc,LCO,0.8012,0.7523,1.6239,0.6319,0.8737,0.9316,0.0053
1,gated+drug_enc,LDO,0.4899,0.4530,2.3138,0.1789,0.7101,1.2024,0.0158
2,gated+drug_enc,LTO,0.8138,0.7719,1.6282,0.6486,0.8806,0.9442,0.0042
3,gated+drug_enc,LPO - all,0.8410,0.8051,1.4797,0.6988,0.8976,0.9533,0.0043
4,gated+drug_enc,LPO - nothing new,0.8751,0.8452,1.3333,0.7604,0.9182,0.9553,0.0041
5,gated+drug_enc,LPO - new cell line only,0.8304,0.7706,1.5129,0.6801,0.8832,0.9275,0.0146
6,gated+drug_enc,LPO - new drug only,0.4967,0.4595,2.2600,0.1676,0.7162,1.2025,0.0473
7,gated+drug_enc,LPO - fully new (cell+drug),0.4208,0.3616,2.5124,0.1026,0.6613,1.2916,0.1886


---
## Combined results

In [ ]:
all_results = pd.concat(
    [arm1_results, arm2_results, arm3_results, arm4_results,
     arm5_results, arm6_results, arm7_results,
     arm8_results, arm9_results],
    ignore_index=True,
)

lco_summary = (
    all_results[all_results['split'] == 'LCO']
    [['arm', 'Pearson r', 'Spearman r', 'RMSE', 'R2', 'ROC-AUC']]
    .sort_values('Pearson r', ascending=False)
    .reset_index(drop=True)
)
print('LCO results:')
display(lco_summary)

print('\nPearson r — all splits:')
display(all_results.pivot_table(index='arm', columns='split', values='Pearson r'))

In [ ]:
all_results.to_csv(RESULTS_DIR / 'concat_multimodal_results_fold0.csv', index=False)
print(f"Saved to {RESULTS_DIR / 'concat_multimodal_results_fold0.csv'}")

## Learning curves

In [ ]:
def plot_learning_curves(histories: dict, metric: str = 'cor') -> None:
    n = len(histories)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=True)
    if n == 1:
        axes = [axes]
    ylabel = 'Pearson r' if metric == 'cor' else 'Loss'
    for ax, (arm_name, hist) in zip(axes, histories.items()):
        ax.plot(hist[f'train_{metric}'], label='train')
        ax.plot(hist[f'val_{metric}'],   label='val')
        ax.set_title(arm_name, fontsize=8)
        ax.set_xlabel('epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7)
    plt.suptitle(f'Learning curves — {ylabel}', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'learning_curves_{metric}.png', dpi=120, bbox_inches='tight')
    plt.show()

# Tier 1 — raw concat
tier1_histories = {
    'rna+protein':         arm1_history,
    'rna+drug_fp':         arm2_history,
    'protein+drug_fp':     arm3_history,
    'rna+protein+drug_fp': arm4_history,
}
plot_learning_curves(tier1_histories, metric='cor')
plot_learning_curves(tier1_histories, metric='loss')

# Tier 2 — encoder-projected concat
tier2_histories = {
    'rna_enc+drug_enc':              arm5_history,
    'protein_enc+drug_enc':          arm6_history,
    'rna_enc+protein_enc+drug_enc':  arm7_history,
}
plot_learning_curves(tier2_histories, metric='cor')
plot_learning_curves(tier2_histories, metric='loss')

---
## Multi-fold evaluation — encoder arms

Arms 5, 6, 7 trained and evaluated on all 5 folds.
Feature selection is refit on each fold's train set.
Fold 0 is re-run here for consistency — earlier fold-0 cells remain for reference.

In [49]:
def run_one_fold(fold_idx: int,
                 model_fn,
                 arm_keys: list,
                 checkpoint_stem: str,
                 fit_fn=None,
                 predict_fn=None) -> pd.DataFrame:
    """Train and evaluate one model on one fold (folds 0-4).
    model_fn    : callable(top_cols) -> nn.Module
    arm_keys    : modality keys for make_dataloader
    fit_fn      : training function (default: fit_encoded_mlp)
    predict_fn  : inference function (default: predict_encoded)
    Returns a tidy DataFrame with a 'fold' column.
    """
    if fit_fn is None:
        fit_fn = fit_encoded_mlp
    if predict_fn is None:
        predict_fn = lambda model, m: predict_encoded(model, m, arm_keys)

    fold = folds[fold_idx]
    train_idx = np.array(fold['train'])

    fold_top_cols = {
        'rna':     select_top_variance_omics('rna',     train_idx, TOP_K_FEATURES),
        'protein': select_top_variance_omics('protein', train_idx, TOP_K_FEATURES),
    }

    inner_idx, val_idx = make_validation_indices(train_idx)

    tr_m  = build_feature_matrix(inner_idx,                        fold_top_cols)
    va_m  = build_feature_matrix(val_idx,                          fold_top_cols)
    lco_m = build_feature_matrix(np.array(fold['lco_test']),       fold_top_cols)
    ldo_m = build_feature_matrix(np.array(fold['ldo_test']),       fold_top_cols)
    lto_m = build_feature_matrix(np.array(fold['lto_test']),       fold_top_cols)
    lpo_m = build_feature_matrix(np.array(fold['lpo_test']),       fold_top_cols)

    cell_ho = set(holdout_groups[fold_idx]['cell_lines_held_out'])
    drug_ho = set(holdout_groups[fold_idx]['drugs_held_out'])
    lpo_idx = np.array(fold['lpo_test'])
    sub     = pairs.loc[lpo_idx]
    is_new_cell = sub[COL_CELLOSAURUS].isin(cell_ho).to_numpy()
    is_new_drug = sub[COL_DRUG].isin(drug_ho).to_numpy()
    fold_lpo_masks = {
        'LPO - nothing new':           (~is_new_cell) & (~is_new_drug),
        'LPO - new cell line only':     is_new_cell   & (~is_new_drug),
        'LPO - new drug only':          (~is_new_cell) & is_new_drug,
        'LPO - fully new (cell+drug)':  is_new_cell   & is_new_drug,
    }

    tr_loader = make_dataloader(tr_m, arm_keys, BATCH_SIZE, shuffle=True)
    va_loader = make_dataloader(va_m, arm_keys, BATCH_SIZE, shuffle=False)

    model = model_fn(fold_top_cols)
    ckpt  = RESULTS_DIR / 'checkpoints' / f'{checkpoint_stem}_fold{fold_idx}.pt'
    model, _ = fit_fn(
        model=model, train_loader=tr_loader, val_loader=va_loader,
        arm_keys=arm_keys, num_epochs=NUM_EPOCHS, patience=PATIENCE,
        checkpoint_path=ckpt, device=DEVICE,
    )

    rows = []
    for split_name, (y_true, y_pred) in [
        ('LCO', (lco_m['y'], predict_fn(model, lco_m))),
        ('LDO', (ldo_m['y'], predict_fn(model, ldo_m))),
        ('LTO', (lto_m['y'], predict_fn(model, lto_m))),
        ('LPO', (lpo_m['y'], predict_fn(model, lpo_m))),
    ]:
        if split_name == 'LPO':
            row = {'fold': fold_idx, 'split': 'LPO - all'}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)
            for mask_name, mask in fold_lpo_masks.items():
                if mask.sum() == 0:
                    continue
                row = {'fold': fold_idx, 'split': mask_name}
                row.update(evaluate_split(y_true[mask], y_pred[mask]))
                rows.append(row)
        else:
            row = {'fold': fold_idx, 'split': split_name}
            row.update(evaluate_split(y_true, y_pred))
            rows.append(row)

    return pd.DataFrame(rows)


def summarise_folds(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby('split')['Pearson r']
        .agg(['mean', 'std'])
        .rename(columns={'mean': 'Pearson r mean', 'std': 'Pearson r std'})
        .round(4)
        .reset_index()
    )

### Arm 5 — `rna_enc + drug_enc` — all folds

In [ ]:
# rna_enc + drug_enc — all folds
arm5_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | rna_enc+drug_enc ---')
    def make_arm5(top_cols):
        return EncodedConcatMLP(
            encoders=[OmicsEncoder(len(top_cols['rna'])), DrugMLPEncoder(DIM_DRUG_FP)],
            n_modalities=2,
        )
    fold_df = run_one_fold(fold_idx, make_arm5, ['rna', 'drug'], 'arm5_rna_enc_drug_enc')
    arm5_fold_results.append(fold_df)

arm5_all_folds = pd.concat(arm5_fold_results, ignore_index=True)
arm5_all_folds.to_csv(RESULTS_DIR / 'arm5_rna_enc_drug_enc_all_folds.csv', index=False)
print('\nrna_enc+drug_enc summary:')
display(summarise_folds(arm5_all_folds))

### Arm 6 — `protein_enc + drug_enc` — all folds

In [ ]:
# protein_enc + drug_enc — all folds
arm6_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | protein_enc+drug_enc ---')
    def make_arm6(top_cols):
        return EncodedConcatMLP(
            encoders=[OmicsEncoder(len(top_cols['protein'])), DrugMLPEncoder(DIM_DRUG_FP)],
            n_modalities=2,
        )
    fold_df = run_one_fold(fold_idx, make_arm6, ['protein', 'drug'], 'arm6_protein_enc_drug_enc')
    arm6_fold_results.append(fold_df)

arm6_all_folds = pd.concat(arm6_fold_results, ignore_index=True)
arm6_all_folds.to_csv(RESULTS_DIR / 'arm6_protein_enc_drug_enc_all_folds.csv', index=False)
print('\nprotein_enc+drug_enc summary:')
display(summarise_folds(arm6_all_folds))

### Arm 7 — `rna_enc + protein_enc + drug_enc` — all folds

In [ ]:
# rna_enc + protein_enc + drug_enc — all folds
arm7_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | rna_enc+protein_enc+drug_enc ---')
    def make_arm7(top_cols):
        return EncodedConcatMLP(
            encoders=[
                OmicsEncoder(len(top_cols['rna'])),
                OmicsEncoder(len(top_cols['protein'])),
                DrugMLPEncoder(DIM_DRUG_FP),
            ],
            n_modalities=3,
        )
    fold_df = run_one_fold(fold_idx, make_arm7, ['rna', 'protein', 'drug'],
                           'arm7_rna_enc_protein_enc_drug_enc')
    arm7_fold_results.append(fold_df)

arm7_all_folds = pd.concat(arm7_fold_results, ignore_index=True)
arm7_all_folds.to_csv(RESULTS_DIR / 'arm7_rna_enc_protein_enc_drug_enc_all_folds.csv', index=False)
print('\nrna_enc+protein_enc+drug_enc summary:')
display(summarise_folds(arm7_all_folds))

### Arm 8 — `hadamard omics fusion` — all folds

In [50]:
# Arm 8 — hadamard omics fusion — all folds
arm8_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | hadamard+drug_enc ---')
    def make_arm8(top_cols):
        return OmicsFusionMLP(
            rna_dim=len(top_cols['rna']),
            protein_dim=len(top_cols['protein']),
            drug_dim=DIM_DRUG_FP,
            fusion_mode='hadamard',
        )
    def fit_fusion_wrapper(model, train_loader, val_loader, arm_keys,
                           num_epochs, patience, checkpoint_path, device):
        return fit_fusion_mlp(model, train_loader, val_loader,
                              arm_name='hadamard+drug_enc',
                              num_epochs=num_epochs, patience=patience,
                              checkpoint_path=checkpoint_path, device=device)
    fold_df = run_one_fold(
        fold_idx, make_arm8, ARM_ALL_KEYS, 'arm8_hadamard',
        fit_fn=fit_fusion_wrapper,
        predict_fn=predict_fusion,
    )
    arm8_fold_results.append(fold_df)

arm8_all_folds = pd.concat(arm8_fold_results, ignore_index=True)
arm8_all_folds.to_csv(RESULTS_DIR / 'arm8_hadamard_all_folds.csv', index=False)
print('\nHadamard summary:')
display(summarise_folds(arm8_all_folds))


--- Fold 0 | hadamard+drug_enc ---
Training arm: hadamard+drug_enc | params: 2,604,801


epoch 1 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch   1 | val loss=3.1181  pearson_r=0.0378


epoch 1 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 2 val:   0%|          | 0/442 [00:00<?, ?it/s]

...

epoch 20 val:   0%|          | 0/421 [00:00<?, ?it/s]

  epoch  20 | val loss=0.8537  pearson_r=0.7662


epoch 20 train:   0%|          | 0/1190 [00:00<?, ?it/s]

  Best val loss: 0.8309 -> /content/drive/MyDrive/multiomics_project/results/concat_multimodal/checkpoints/arm8_hadamard_fold4.pt

Hadamard summary:


,split,Pearson r mean,Pearson r std
0,LCO,0.7967,0.0212
1,LDO,0.3727,0.0729
2,LPO - all,0.8271,0.0188
3,LPO - fully new (cell+drug),0.2999,0.1410
4,LPO - new cell line only,0.8342,0.0119
5,LPO - new drug only,0.3871,0.0838
6,LPO - nothing new,0.8700,0.0061
7,LTO,0.7968,0.0322


### Arm 9 — `gated omics fusion` — all folds

In [51]:
# Arm 9 — gated omics fusion — all folds
arm9_fold_results = []

for fold_idx in range(5):
    print(f'\n--- Fold {fold_idx} | gated+drug_enc ---')
    def make_arm9(top_cols):
        return OmicsFusionMLP(
            rna_dim=len(top_cols['rna']),
            protein_dim=len(top_cols['protein']),
            drug_dim=DIM_DRUG_FP,
            fusion_mode='gated',
        )
    def fit_fusion_wrapper(model, train_loader, val_loader, arm_keys,
                           num_epochs, patience, checkpoint_path, device):
        return fit_fusion_mlp(model, train_loader, val_loader,
                              arm_name='gated+drug_enc',
                              num_epochs=num_epochs, patience=patience,
                              checkpoint_path=checkpoint_path, device=device)
    fold_df = run_one_fold(
        fold_idx, make_arm9, ARM_ALL_KEYS, 'arm9_gated',
        fit_fn=fit_fusion_wrapper,
        predict_fn=predict_fusion,
    )
    arm9_fold_results.append(fold_df)

arm9_all_folds = pd.concat(arm9_fold_results, ignore_index=True)
arm9_all_folds.to_csv(RESULTS_DIR / 'arm9_gated_all_folds.csv', index=False)
print('\nGated summary:')
display(summarise_folds(arm9_all_folds))


--- Fold 0 | gated+drug_enc ---
Training arm: gated+drug_enc | params: 2,735,873


epoch 1 val:   0%|          | 0/442 [00:00<?, ?it/s]

  epoch   1 | val loss=3.0769  pearson_r=0.0569


epoch 1 train:   0%|          | 0/1237 [00:00<?, ?it/s]

epoch 2 val:   0%|          | 0/442 [00:00<?, ?it/s]

...

  epoch  18 | val loss=0.8595  pearson_r=0.7665


epoch 18 train:   0%|          | 0/1190 [00:00<?, ?it/s]

epoch 19 val:   0%|          | 0/421 [00:00<?, ?it/s]

  epoch  19 | val loss=0.8664  pearson_r=0.7641
  Early stopping at epoch 19
  Best val loss: 0.8493 -> /content/drive/MyDrive/multiomics_project/results/concat_multimodal/checkpoints/arm9_gated_fold4.pt

Gated summary:


,split,Pearson r mean,Pearson r std
0,LCO,0.8002,0.0156
1,LDO,0.3826,0.0452
2,LPO - all,0.8315,0.0150
3,LPO - fully new (cell+drug),0.3100,0.1159
4,LPO - new cell line only,0.8390,0.0075
5,LPO - new drug only,0.3934,0.0619
6,LPO - nothing new,0.8738,0.0024
7,LTO,0.8005,0.0257


### Combined summary

In [52]:
arm5_all_folds['arm'] = 'rna_enc+drug_enc'
arm6_all_folds['arm'] = 'protein_enc+drug_enc'
arm7_all_folds['arm'] = 'rna_enc+protein_enc+drug_enc'
arm8_all_folds['arm'] = 'hadamard+drug_enc'
arm9_all_folds['arm'] = 'gated+drug_enc'

all_folds_combined = pd.concat(
    [arm5_all_folds, arm6_all_folds, arm7_all_folds,
     arm8_all_folds, arm9_all_folds],
    ignore_index=True,
)
all_folds_combined.to_csv(RESULTS_DIR / 'encoder_arms_all_folds.csv', index=False)



def mean_std_str(s: pd.Series) -> str:
    return f"{s.mean():.3f} ± {s.std():.3f}"
summary = (
    all_folds_combined
    .groupby(['arm', 'split'])[["Pearson r", "Spearman r", "RMSE", "R2", "ROC-AUC"]]
    .agg(mean_std_str)
    .round(4)
    .rename(columns={'mean': 'mean', 'std': 'std'})
)

display(summary)

Pearson r  \
arm                          split                                        
gated+drug_enc               LCO                          0.800 ± 0.016   
                             LDO                          0.383 ± 0.045   
                             LPO - all                    0.831 ± 0.015   
                             LPO - fully new (cell+drug)  0.310 ± 0.116   
                             LPO - new cell line only     0.839 ± 0.007   
                             LPO - new drug only          0.393 ± 0.062   
                             LPO - nothing new            0.874 ± 0.002   
                             LTO                          0.800 ± 0.026   
hadamard+drug_enc            LCO                          0.797 ± 0.021   
                             LDO                          0.373 ± 0.073   
                             LPO - all                    0.827 ± 0.019   
                             LPO - fully new (cell+drug)  0.300 ± 0.141   
                             LPO - new cell line only     0.834 ± 0.012   
                             LPO - new drug only          0.387 ± 0.084   
                             LPO - nothing new            0.870 ± 0.006   
                             LTO                          0.797 ± 0.032   
protein_enc+drug_enc         LCO                          0.796 ± 0.019   
                             LDO                          0.368 ± 0.069   
                             LPO - all                    0.829 ± 0.017   
                             LPO - fully new (cell+drug)  0.287 ± 0.140   
                             LPO - new cell line only     0.836 ± 0.008   
                             LPO - new drug only          0.380 ± 0.075   
                             LPO - nothing new            0.873 ± 0.004   
                             LTO                          0.796 ± 0.027   
rna_enc+drug_enc             LCO                          0.795 ± 0.021   
                             LDO                          0.352 ± 0.080   
                             LPO - all                    0.825 ± 0.019   
                             LPO - fully new (cell+drug)  0.274 ± 0.139   
                             LPO - new cell line only     0.832 ± 0.009   
                             LPO - new drug only          0.361 ± 0.089   
                             LPO - nothing new            0.868 ± 0.007   
                             LTO                          0.797 ± 0.030   
rna_enc+protein_enc+drug_enc LCO                          0.801 ± 0.017   
                             LDO                          0.368 ± 0.081   
                             LPO - all                    0.831 ± 0.013   
                             LPO - fully new (cell+drug)  0.277 ± 0.143   
                             LPO - new cell line only     0.839 ± 0.009   
                             LPO - new drug only          0.381 ± 0.092   
                             LPO - nothing new            0.874 ± 0.003   
                             LTO                          0.800 ± 0.026   

                                                             Spearman r  \
arm                          split                                        
gated+drug_enc               LCO                          0.748 ± 0.015   
                             LDO                          0.351 ± 0.063   
                             LPO - all                    0.792 ± 0.013   
                             LPO - fully new (cell+drug)  0.270 ± 0.084   
                             LPO - new cell line only     0.794 ± 0.010   
                             LPO - new drug only          0.363 ± 0.088   
                             LPO - nothing new            0.841 ± 0.005   
                             LTO                          0.752 ± 0.027   
hadamard+drug_enc            LCO                          0.745 ± 0.020   
                             LDO                          0.342 ± 0.083   
           